# 🚀 DBT Models + Jinja Templating (Complete Study Notes)

------------------------------------------------------------------------

# 🧠 1. What is a DBT Model?

A DBT model is: - A SQL file - Located in `models/` - Compiled into a
table/view in warehouse

👉 With Jinja: SQL + Jinja = Dynamic transformation

------------------------------------------------------------------------

# 🧩 2. Types of Models (Layered Architecture)

## 🧹 Staging Layer

-   Clean raw data
-   Rename columns

``` sql
{{ config(materialized='view') }}

SELECT
    id,
    customer_id,
    amount,
    created_at
FROM {{ source('raw', 'orders') }}
```

------------------------------------------------------------------------

## 🔄 Intermediate Layer

-   Business logic
-   Aggregations

``` sql
SELECT
    customer_id,
    COUNT(*) AS total_orders,
    SUM(amount) AS total_spent
FROM {{ ref('stg_orders') }}
GROUP BY customer_id
```

------------------------------------------------------------------------

## 📊 Mart Layer

-   Final analytics tables

``` sql
SELECT
    customer_id,
    total_orders,
    total_spent,
    CASE
        WHEN total_spent > 10000 THEN 'High Value'
        ELSE 'Low Value'
    END AS segment
FROM {{ ref('int_orders') }}
```

------------------------------------------------------------------------

# 🔗 3. DAG Using ref()

``` sql
SELECT * FROM {{ ref('stg_orders') }}
```

Flow: raw → staging → intermediate → mart

------------------------------------------------------------------------

# ⚙️ 4. Materializations

``` sql
{{ config(materialized='table') }}
```

Types: - view - table - incremental

------------------------------------------------------------------------

# 🔥 5. Incremental Model (Production)

``` sql
{{ config(materialized='incremental') }}

SELECT *
FROM {{ ref('stg_orders') }}

{% if is_incremental() %}
WHERE updated_at > (SELECT MAX(updated_at) FROM {{ this }})
{% endif %}
```

------------------------------------------------------------------------

# 🧠 6. Jinja Core Concepts

## ref()

``` sql
{{ ref('model_name') }}
```

## source()

``` sql
{{ source('raw', 'orders') }}
```

## config()

``` sql
{{ config(materialized='table') }}
```

## this

``` sql
SELECT * FROM {{ this }}
```

------------------------------------------------------------------------

# 🔁 7. Jinja Advanced Usage

## Conditional Logic

``` sql
{% if target.name == 'dev' %}
LIMIT 100
{% endif %}
```

------------------------------------------------------------------------

## Variables

``` sql
WHERE country = '{{ var("country", "IN") }}'
```

------------------------------------------------------------------------

## Loops

``` sql
{% for col in ['id','amount'] %}
    {{ col }}{% if not loop.last %},{% endif %}
{% endfor %}
```

------------------------------------------------------------------------

# 🧠 8. Macros (Reusable Logic)

``` sql
{% macro segment(amount) %}
CASE
    WHEN {{ amount }} > 10000 THEN 'High'
    ELSE 'Low'
END
{% endmacro %}
```

Usage:

``` sql
SELECT {{ segment('total_spent') }}
```

------------------------------------------------------------------------

# 🏗️ 9. Production-Level Example

## Staging

``` sql
SELECT * FROM {{ source('raw', 'orders') }}
```

## Intermediate

``` sql
SELECT
    customer_id,
    SUM(amount) AS total_spent
FROM {{ ref('stg_orders') }}
GROUP BY customer_id
```

## Mart (Incremental)

``` sql
{{ config(materialized='incremental', unique_key='customer_id') }}

SELECT *
FROM {{ ref('int_orders') }}

{% if is_incremental() %}
WHERE updated_at > (SELECT MAX(updated_at) FROM {{ this }})
{% endif %}
```

------------------------------------------------------------------------

# 🧪 10. Testing

``` yaml
models:
  - name: stg_orders
    columns:
      - name: id
        tests:
          - unique
          - not_null
```

------------------------------------------------------------------------

# ⚙️ 11. Internal Working

1.  Parse models\
2.  Apply Jinja\
3.  Build DAG\
4.  Compile SQL\
5.  Execute in warehouse

------------------------------------------------------------------------

# 🚀 12. Best Practices

-   Use staging → intermediate → marts\
-   Always use ref()\
-   Avoid hardcoding\
-   Use incremental for large data\
-   Use macros for reuse

------------------------------------------------------------------------

# 🎯 13. Interview Points

-   Model = SQL file\
-   Jinja = dynamic SQL\
-   ref() = DAG\
-   this = current model\
-   Incremental = performance

------------------------------------------------------------------------

# ⚡ Final Summary

DBT Models: - Modular SQL transformations\
- Jinja-powered dynamic logic\
- Scalable via warehouse

👉 Core Idea: DBT = SQL + Jinja + DAG + Warehouse Execution


# 🚀 DBT Property Hierarchy --- Complete Detailed Notes (With Overwrite Examples)

------------------------------------------------------------------------

# 🧠 1. What is Property Hierarchy in DBT?

DBT allows configuration at multiple levels:

-   dbt_project.yml (global / folder level)
-   properties.yml (model level)
-   config() block (inside SQL model)

👉 When the same property is defined in multiple places, DBT follows a
**priority hierarchy**.

------------------------------------------------------------------------

# 🏆 2. Priority Order (Highest → Lowest)

config() (SQL) \> properties.yml \> dbt_project.yml

👉 Highest priority wins

------------------------------------------------------------------------

# 🧩 3. Levels Explained

------------------------------------------------------------------------

## 🔹 3.1 dbt_project.yml (Lowest Priority)

-   Global defaults
-   Folder-level configuration

``` yaml
models:
  my_project:
    staging:
      materialized: view
```

👉 All staging models → view (default)

------------------------------------------------------------------------

## 🔹 3.2 properties.yml (Middle Layer)

-   Model-specific overrides
-   Also used for tests & documentation

``` yaml
models:
  - name: stg_orders
    config:
      materialized: table
```

👉 Overrides project-level config

------------------------------------------------------------------------

## 🔹 3.3 config() Block (Highest Priority)

-   Inside SQL model
-   Final override

``` sql
{{ config(materialized='incremental') }}

SELECT * FROM raw.orders
```

👉 Overrides everything

------------------------------------------------------------------------

# 🔥 4. Full Overwrite Example

------------------------------------------------------------------------

## Step 1: dbt_project.yml

``` yaml
models:
  my_project:
    staging:
      materialized: view
```

------------------------------------------------------------------------

## Step 2: properties.yml

``` yaml
models:
  - name: stg_orders
    config:
      materialized: table
```

------------------------------------------------------------------------

## Step 3: SQL Model

``` sql
{{ config(materialized='incremental') }}

SELECT * FROM raw.orders
```

------------------------------------------------------------------------

## 🧠 Final Result

materialized = incremental

------------------------------------------------------------------------

## 📊 Override Flow

dbt_project.yml (view) ↓ properties.yml (table) ↓ config() (incremental)
✅ FINAL

------------------------------------------------------------------------

# 🧪 5. Multi-Property Production Example

------------------------------------------------------------------------

## dbt_project.yml

``` yaml
models:
  my_project:
    marts:
      materialized: table
      schema: analytics
```

------------------------------------------------------------------------

## properties.yml

``` yaml
models:
  - name: mart_customer_summary
    config:
      materialized: incremental
      schema: business
```

------------------------------------------------------------------------

## SQL Model

``` sql
{{ config(
    materialized='view',
    schema='final_layer'
) }}

SELECT * FROM {{ ref('int_orders') }}
```

------------------------------------------------------------------------

## 🧠 Final Output

  Property       Final Value   Source
  -------------- ------------- ----------
  materialized   view          config()
  schema         final_layer   config()

------------------------------------------------------------------------

# 🧠 6. Partial Override Example

------------------------------------------------------------------------

## dbt_project.yml

``` yaml
models:
  my_project:
    staging:
      materialized: view
      schema: staging_layer
```

------------------------------------------------------------------------

## properties.yml

``` yaml
models:
  - name: stg_orders
    config:
      materialized: table
```

------------------------------------------------------------------------

## SQL Model

``` sql
{{ config(schema='custom_schema') }}

SELECT * FROM raw.orders
```

------------------------------------------------------------------------

## 🧠 Final Output

  Property       Final Value     Source
  -------------- --------------- ----------------
  materialized   table           properties.yml
  schema         custom_schema   config()

------------------------------------------------------------------------

👉 Only overwritten properties change\
👉 Others are inherited from lower levels

------------------------------------------------------------------------

# 🔁 7. Visual Hierarchy

dbt_project.yml\
↓\
properties.yml\
↓\
config()

------------------------------------------------------------------------

# ⚠️ 8. Important Notes

-   Avoid overusing config() (reduces readability)
-   Use project file for defaults
-   Use properties.yml for metadata + tests
-   Use config() for dynamic or critical overrides

------------------------------------------------------------------------

# 🚀 9. Best Practices (Production)

-   Define defaults in dbt_project.yml\
-   Override per model in properties.yml\
-   Use config() only when necessary\
-   Keep configs clean and consistent

------------------------------------------------------------------------

# 🎯 10. Interview Answer

👉 DBT property hierarchy:

config() \> properties.yml \> dbt_project.yml

👉 Only overridden properties change, others are inherited.

------------------------------------------------------------------------

# ⚡ Final Summary

-   DBT supports multi-level configuration\
-   Follows strict priority hierarchy\
-   Enables flexible and scalable pipeline design


# 🚀 DBT: Run Only Selected Models (Selective Execution)

------------------------------------------------------------------------

## 🧠 Basic Syntax

``` bash
dbt run --select <model_name>
```

Shortcut:

``` bash
dbt run -s <model_name>
```

------------------------------------------------------------------------

## 🧩 1. Run Single Model

``` bash
dbt run -s stg_orders
```

------------------------------------------------------------------------

## 🧩 2. Run Multiple Models

``` bash
dbt run -s stg_orders stg_customers
```

------------------------------------------------------------------------

## 🧩 3. Run Models with Dependencies (Upstream)

``` bash
dbt run -s +mart_customer_summary
```

------------------------------------------------------------------------

## 🧩 4. Run Models with Downstream

``` bash
dbt run -s stg_orders+
```

------------------------------------------------------------------------

## 🧩 5. Full DAG Around Model

``` bash
dbt run -s +stg_orders+
```

------------------------------------------------------------------------

## 🧩 6. Run by Folder

``` bash
dbt run -s staging
```

------------------------------------------------------------------------

## 🧩 7. Run by Path

``` bash
dbt run -s models/staging
```

------------------------------------------------------------------------

## 🧩 8. Run by Tag

``` sql
{{ config(tags=['finance']) }}
```

``` bash
dbt run -s tag:finance
```

------------------------------------------------------------------------

## 🧩 9. Run by Wildcard

``` bash
dbt run -s stg_*
```

------------------------------------------------------------------------

## 🧩 10. Exclude Models

``` bash
dbt run -s staging --exclude stg_orders
```

------------------------------------------------------------------------

## 🧩 11. Combine Selection

``` bash
dbt run -s staging,tag:finance
```

------------------------------------------------------------------------

## 🧩 12. Run Modified Models

``` bash
dbt run -s state:modified
```

------------------------------------------------------------------------

## 🧠 Production Examples

-   Debug one model\
-   Run full pipeline\
-   Run staging layer\
-   CI/CD optimized runs

------------------------------------------------------------------------

## 🎯 Summary

-   Use -s for selection\
-   Use + for dependencies\
-   Use tags and state for flexibility
